<a href="https://colab.research.google.com/github/AktanM11/AI-OI/blob/main/DAY3_assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install langgraph

In [14]:
%pip install langchain-groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 7.9 MB/s eta 0:00:00


In [15]:
import os
from google.colab import userdata
from typing import Literal
from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.prebuilt import ToolNode
import time

os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')

@tool
def multiplication(a: float, b: float) -> float:
    """Multipliy a by b"""
    print(f"-> Tool Executed [multiplication]: {a} * {b}")
    return a * b

@tool
def substraction(a: float, b: float) -> float:
    """Subtract a by b"""
    print(f"-> Tool Executed [substraction]: {a} - {b}")
    return a - b

@tool
def calculate_factorial(n: int) -> int:
    """Factorial of n"""
    print(f"-> Tool Executed [calculate_factorial]: {n}!")
    import math
    return math.factorial(int(n))

my_clean_tools_list = [multiplication, substraction, calculate_factorial,]
fresh_tool_node = ToolNode(my_clean_tools_list)

model = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0
)
model_with_tools = model.bind_tools(my_clean_tools_list)

def agent_reasoning_node(state: MessagesState):
    time.sleep(2.0)
    response = model_with_tools.invoke(state["messages"])
    return {"messages": [response]}

def routing_edge_decision(state: MessagesState) -> Literal["execute_tools", "__end__"]:
    last_msg = state["messages"][-1]
    if last_msg.tool_calls:
        return "execute_tools"
    return "__end__"


new_workflow = StateGraph(MessagesState)

# Map our fresh nodes
new_workflow.add_node("call_ai", agent_reasoning_node)
new_workflow.add_node("execute_tools", fresh_tool_node)

# Set up edges
new_workflow.add_edge(START, "call_ai")
new_workflow.add_conditional_edges("call_ai", routing_edge_decision)
new_workflow.add_edge("execute_tools", "call_ai")

# Compile fresh instance
compiled_graph = new_workflow.compile()
print("Success! The clean graph has compiled successfully.")

Success! The clean graph has compiled successfully.


In [16]:
problem_statement = "Take the number 45, multiply it by 3, subtract 131 and calculate the factorial of that resul"

test_state = {"messages": [("user", problem_statement)]}
output_results = compiled_graph.invoke(test_state)

print("AGENT FINAL ANSWER:")
print(output_results["messages"][-1].content)


-> Tool Executed [multiplication]: 45.0 * 3.0
-> Tool Executed [substraction]: 135.0 - 131.0
-> Tool Executed [calculate_factorial]: 4!
AGENT FINAL ANSWER:
The result of the operations is 24. 

First, 45 was multiplied by 3, resulting in 135. Then, 131 was subtracted from 135, resulting in 4. Finally, the factorial of 4 was calculated, which is 4 * 3 * 2 * 1 = 24.
